# Reprodução do PPI e dos painéis de avaliação

Este notebook verifica o conjunto publicado, reproduz os candidatos PPI a partir das séries WTSS incluídas, compara o resultado com os candidatos de referência e gera uma nova cópia dos painéis. Todos os caminhos são relativos ao repositório.

In [ ]:
from pathlib import Path
import pandas as pd

from worcap_endmembers.config import find_repository, load_config
from worcap_endmembers.panel import build_panels
from worcap_endmembers.workflow import compare_candidates, run_ppi, verify_release

ROOT = find_repository(Path.cwd())
ROOT

## 1. Verificação da versão pública

A verificação confere 5.000 pontos, 1.000 por classe, 184 datas WTSS por ponto, manifestos PPI completos e os checksums da versão.

In [ ]:
verification = verify_release(ROOT)
verification

## 2. Amostra sistemática

A amostra está disponível em CSV, Parquet e GeoPackage. O resumo abaixo utiliza o Parquet portátil.

In [ ]:
points = pd.read_parquet(ROOT / 'data' / 'points' / 'points.parquet')
points['class_code'] = points['class_code'].astype(str).str.zfill(2)
points.groupby('class_code').agg(
    points=('point_id', 'size'),
    minimum_distance_km=('class_min_distance_m', lambda x: x.iloc[0] / 1000),
)

## 3. Reprodução dos candidatos PPI

O PPI é executado separadamente para cada classe e data com 10.000 projeções, seed 13, máximo de quatro candidatos e separação SAM mínima de 2°. O processamento utiliza os Parquets WTSS incluídos, sem novas chamadas de rede.

In [ ]:
ppi_summary = run_ppi(ROOT, output='outputs/notebook_reproduction')
pd.DataFrame(ppi_summary)

## 4. Comparação com a versão publicada

A comparação exige igualdade dos identificadores, pixels selecionados, escores PPI e valores das dez bandas.

In [ ]:
comparison = compare_candidates(
    'outputs/notebook_reproduction/candidates',
    'data/candidates',
    ROOT,
)
comparison

## 5. Geração dos painéis

Os painéis reproduzidos são gravados em `outputs/notebook_reproduction/panels/`. As escolhas permanecem no armazenamento local do navegador e são exportadas em CSV.

In [ ]:
panel_summary = build_panels(
    ROOT,
    candidates='outputs/notebook_reproduction/candidates',
    output='outputs/notebook_reproduction/panels',
)
panel_summary

## 6. Avaliações

Após a revisão, salve cada CSV em `reviews/incoming/class_XX/` e execute:

```bash
python -m worcap_endmembers validate-all-reviews
python -m worcap_endmembers merge-reviews
```